In [1]:
from langchain_core.tools import tool
import requests

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os

api_key = os.getenv("AICREDITS_API_KEY")
base_url = os.getenv("AICREDITS_BASE_URL")


In [3]:
from langchain_core.tools import tool
import json

@tool
def query_wolfram_alpha(expression: str) -> str:
    """Query Wolfram Alpha to compute expressions or retrieve information.
    
    Args: 
        expression (str): The mathematical expression or query to evaluate.
    Returns: 
        str: The result of the computation or the retrieved information.
    """
    print(f"🔧 [MOCK WOLFRAM] Evaluating expression: '{expression}'")
    
    # Optional: Basic local evaluator if the model passes clean math strings
    try:
        # Cleans common wording out if the model asks a conversational math question
        clean_expr = expression.lower().replace("what is", "").replace("?", "").strip()
        # Handle a few common textbook string formulas safely
        if "^" in clean_expr: 
            clean_expr = clean_expr.replace("^", "**")
            
        # Safely evaluate simple math strings locally
        calc_result = eval(clean_expr, {"__builtins__": None}, {})
        return f"Result: {calc_result}"
    except Exception:
        # Clever realistic fallback answer for complex queries
        return f"Wolfram Alpha computed result for query '{expression}': operation completed successfully."


@tool
def trigger_zapier_webhook(zap_id: str, payload: dict) -> str:
    """Trigger a Zapier webhook to execute a predefined Zap.
    
    Args:
        zap_id (str): The unique identifier for the Zap to be triggered.
        payload (dict): The data to send to the Zapier webhook.
    Returns:
        str: Confirmation message upon successful triggering of the Zap.
    """
    print(f"⚡ [MOCK ZAPIER] Webhook intercept triggered!")
    print(f"   ↳ Zap ID: {zap_id}")
    print(f"   ↳ Payload Data Sent: {json.dumps(payload, indent=2)}")
    
    return f"Zapier webhook '{zap_id}' successfully triggered (Mock Mode)."


@tool
def send_slack_message(channel: str, message: str) -> str:
    """Send a message to a specified Slack channel.
    
    Args:
        channel (str): The Slack channel ID or name where the message will be sent.
        message (str): The content of the message to send.
    Returns:
        str: Confirmation message upon successful sending of the Slack message.
    """
    print(f"💬 [MOCK SLACK] Outbound Message Dispatched:")
    print(f"   ↳ Destination Channel: {channel}")
    print(f"   ↳ Text Body: \"{message}\"")
    
    return f"Message successfully sent to Slack Channel '{channel}'."


In [4]:
llm = ChatOpenAI(model_name= "gpt-4o", api_key=api_key, base_url=base_url)
llm_with_tools = llm.bind_tools([query_wolfram_alpha, trigger_zapier_webhook, send_slack_message])

In [5]:
messages = [HumanMessage("Solve the equation 2x^2-32=0")]

In [6]:
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_msg = query_wolfram_alpha.invoke(tool_call)
    messages.append(tool_msg)

final_response = llm_with_tools.invoke(messages)
print(final_response.content)

🔧 [MOCK WOLFRAM] Evaluating expression: 'solve 2x^2 - 32 = 0'
The solution to the equation \(2x^2 - 32 = 0\) is \(x = \pm 4\).
